In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 01:
-- Validate Gold Analytical Layer
--
-- Purpose:
-- Confirm that the completed Gold warehouse is accessible
-- before beginning business-oriented SQL analysis.

SHOW TABLES IN workspace.gold_sop;

database,tableName,isTemporary
gold_sop,bridge_alternative_machine,false
gold_sop,bridge_product_packing,false
gold_sop,bridge_product_production,false
gold_sop,bridge_product_raw_material,false
gold_sop,dim_machine,false
gold_sop,dim_packing_machine,false
gold_sop,dim_planning_period,false
gold_sop,dim_product,false
gold_sop,fact_customer_order,false
gold_sop,fact_demand_plan,false


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 01A:
-- Validate Gold Table Record Counts
--
-- Purpose:
-- Confirm that all persisted Gold tables retain the
-- expected record counts before business analysis begins.

SELECT 'dim_product' AS table_name, COUNT(*) AS row_count
FROM workspace.gold_sop.dim_product

UNION ALL

SELECT 'dim_planning_period', COUNT(*)
FROM workspace.gold_sop.dim_planning_period

UNION ALL

SELECT 'dim_machine', COUNT(*)
FROM workspace.gold_sop.dim_machine

UNION ALL

SELECT 'dim_packing_machine', COUNT(*)
FROM workspace.gold_sop.dim_packing_machine

UNION ALL

SELECT 'fact_customer_order', COUNT(*)
FROM workspace.gold_sop.fact_customer_order

UNION ALL

SELECT 'fact_demand_plan', COUNT(*)
FROM workspace.gold_sop.fact_demand_plan

UNION ALL

SELECT 'fact_machine_capacity', COUNT(*)
FROM workspace.gold_sop.fact_machine_capacity

UNION ALL

SELECT 'fact_packing_capacity', COUNT(*)
FROM workspace.gold_sop.fact_packing_capacity

UNION ALL

SELECT 'bridge_product_production', COUNT(*)
FROM workspace.gold_sop.bridge_product_production

UNION ALL

SELECT 'bridge_alternative_machine', COUNT(*)
FROM workspace.gold_sop.bridge_alternative_machine

UNION ALL

SELECT 'bridge_product_packing', COUNT(*)
FROM workspace.gold_sop.bridge_product_packing

UNION ALL

SELECT 'bridge_product_raw_material', COUNT(*)
FROM workspace.gold_sop.bridge_product_raw_material

ORDER BY table_name;

table_name,row_count
bridge_alternative_machine,734
bridge_product_packing,189
bridge_product_production,592
bridge_product_raw_material,292
dim_machine,39
dim_packing_machine,7
dim_planning_period,8
dim_product,292
fact_customer_order,61
fact_demand_plan,246


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 02:
-- Demand Planning Overview
--
-- Business Question:
-- How is planned demand distributed across the
-- eight-period planning horizon?
--
-- SQL Skills:
-- JOIN, SUM, COUNT(DISTINCT), GROUP BY, ORDER BY

SELECT
    dp.period,
    pp.macroperiod_month,
    pp.number_of_days,

    COUNT(DISTINCT dp.product) AS products_with_demand,

    SUM(dp.adjusted_total_demand_m) AS total_planned_demand_m,

    SUM(dp.target_stock_m) AS total_target_stock_m

FROM workspace.gold_sop.fact_demand_plan AS dp

INNER JOIN workspace.gold_sop.dim_planning_period AS pp
    ON dp.period = pp.period

GROUP BY
    dp.period,
    pp.macroperiod_month,
    pp.number_of_days

ORDER BY
    dp.period;

period,macroperiod_month,number_of_days,products_with_demand,total_planned_demand_m,total_target_stock_m
3,2,6,41,175527,622136
4,2,7,41,204783,622136
5,2,7,41,204783,622136
6,2,9,41,263293,622136
7,3,28,41,1092148,622136
8,4,29,41,1131158,622136


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 02A:
-- Investigate Planning-Period Coverage
--
-- Business Question:
-- Which planning periods contain demand-plan records,
-- and which periods have no corresponding demand records?
--
-- Purpose:
-- Explain why Step 02 returned only Periods 3–8.
--
-- SQL Skills:
-- LEFT JOIN, COUNT, COUNT(DISTINCT), COALESCE,
-- GROUP BY, ORDER BY

SELECT
    pp.period,
    pp.macroperiod_month,
    pp.number_of_days,

    COUNT(dp.product) AS demand_plan_rows,

    COUNT(DISTINCT dp.product) AS products_with_demand,

    COALESCE(
        SUM(dp.adjusted_total_demand_m),
        0
    ) AS total_planned_demand_m,

    COALESCE(
        SUM(dp.target_stock_m),
        0
    ) AS total_target_stock_m

FROM workspace.gold_sop.dim_planning_period AS pp

LEFT JOIN workspace.gold_sop.fact_demand_plan AS dp
    ON pp.period = dp.period

GROUP BY
    pp.period,
    pp.macroperiod_month,
    pp.number_of_days

ORDER BY
    pp.period;

period,macroperiod_month,number_of_days,demand_plan_rows,products_with_demand,total_planned_demand_m,total_target_stock_m
1,1,7,0,0,0,0
2,1,8,0,0,0,0
3,2,6,41,41,175527,622136
4,2,7,41,41,204783,622136
5,2,7,41,41,204783,622136
6,2,9,41,41,263293,622136
7,3,28,41,41,1092148,622136
8,4,29,41,41,1131158,622136


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 03:
-- Demand Intensity by Planning Period
--
-- Business Question:
-- How much planned demand occurs per planning day
-- after accounting for differences in period length?
--
-- SQL Skills:
-- LEFT JOIN, SUM, COUNT, COALESCE,
-- CASE, ROUND, GROUP BY, ORDER BY

SELECT
    pp.period,
    pp.macroperiod_month,
    pp.number_of_days,

    COUNT(dp.product) AS demand_plan_rows,

    COALESCE(
        SUM(dp.adjusted_total_demand_m),
        0
    ) AS total_planned_demand_m,

    CASE
        WHEN COUNT(dp.product) = 0 THEN 0
        ELSE ROUND(
            SUM(dp.adjusted_total_demand_m)
            / pp.number_of_days,
            2
        )
    END AS planned_demand_per_day_m,

    COALESCE(
        SUM(dp.target_stock_m),
        0
    ) AS total_target_stock_m

FROM workspace.gold_sop.dim_planning_period AS pp

LEFT JOIN workspace.gold_sop.fact_demand_plan AS dp
    ON pp.period = dp.period

GROUP BY
    pp.period,
    pp.macroperiod_month,
    pp.number_of_days

ORDER BY
    pp.period;

period,macroperiod_month,number_of_days,demand_plan_rows,total_planned_demand_m,planned_demand_per_day_m,total_target_stock_m
1,1,7,0,0,0.0,0
2,1,8,0,0,0.0,0
3,2,6,41,175527,29254.5,622136
4,2,7,41,204783,29254.71,622136
5,2,7,41,204783,29254.71,622136
6,2,9,41,263293,29254.78,622136
7,3,28,41,1092148,39005.29,622136
8,4,29,41,1131158,39005.45,622136


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 04:
-- Product-Level Demand Ranking
--
-- Business Question:
-- Which products account for the highest total planned
-- demand across the represented demand-planning periods?
--
-- SQL Skills:
-- JOIN, SUM, COUNT, ROUND,
-- GROUP BY, ORDER BY, LIMIT

SELECT
    dp.product,

    p.mts_mto,

    COUNT(DISTINCT dp.period) AS demand_periods,

    SUM(dp.adjusted_total_demand_m) AS total_planned_demand_m,

    ROUND(
        AVG(dp.adjusted_total_demand_m),
        2
    ) AS avg_demand_per_active_period_m,

    SUM(dp.target_stock_m) AS cumulative_target_stock_m

FROM workspace.gold_sop.fact_demand_plan AS dp

LEFT JOIN workspace.gold_sop.dim_product AS p
    ON dp.product = p.product

GROUP BY
    dp.product,
    p.mts_mto

ORDER BY
    total_planned_demand_m DESC

LIMIT 10;

product,mts_mto,demand_periods,total_planned_demand_m,avg_demand_per_active_period_m,cumulative_target_stock_m
7222002,MTS,6,350530,58421.67,493692
6209061,MTS,6,282207,47034.5,244302
6182513,MTS,6,243147,40524.5,237198
6225112,MTS,6,242800,40466.67,64746
6182514,MTS,6,178270,29711.67,184434
6225111,MTS,6,168256,28042.67,35226
7222001,MTS,6,150076,25012.67,85464
6201036,MTS,6,130884,21814.0,78870
6182516,MTS,6,129196,21532.67,150888
6182512,MTS,6,107878,17979.67,121398


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 05:
-- Demand Concentration Analysis
--
-- Business Question:
-- Which products contribute the most to total planned demand,
-- and what share of overall planned demand does each represent?
--
-- SQL Skills:
-- CTE, SUM, window functions, RANK,
-- percentage calculation, ORDER BY

WITH product_demand AS (

    SELECT
        dp.product,
        p.mts_mto,
        SUM(dp.adjusted_total_demand_m) AS total_planned_demand_m

    FROM workspace.gold_sop.fact_demand_plan AS dp

    LEFT JOIN workspace.gold_sop.dim_product AS p
        ON dp.product = p.product

    GROUP BY
        dp.product,
        p.mts_mto
),

ranked_demand AS (

    SELECT
        product,
        mts_mto,
        total_planned_demand_m,

        RANK() OVER (
            ORDER BY total_planned_demand_m DESC
        ) AS demand_rank,

        ROUND(
            100.0 * total_planned_demand_m
            / SUM(total_planned_demand_m) OVER (),
            2
        ) AS demand_share_pct,

        ROUND(
            100.0 * SUM(total_planned_demand_m) OVER (
                ORDER BY total_planned_demand_m DESC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            )
            / SUM(total_planned_demand_m) OVER (),
            2
        ) AS cumulative_demand_share_pct

    FROM product_demand
)

SELECT
    demand_rank,
    product,
    mts_mto,
    total_planned_demand_m,
    demand_share_pct,
    cumulative_demand_share_pct

FROM ranked_demand

ORDER BY demand_rank

LIMIT 10;

demand_rank,product,mts_mto,total_planned_demand_m,demand_share_pct,cumulative_demand_share_pct
1,7222002,MTS,350530,11.41,11.41
2,6209061,MTS,282207,9.19,20.60
3,6182513,MTS,243147,7.92,28.51
4,6225112,MTS,242800,7.90,36.42
5,6182514,MTS,178270,5.80,42.22
6,6225111,MTS,168256,5.48,47.70
7,7222001,MTS,150076,4.89,52.59
8,6201036,MTS,130884,4.26,56.85
9,6182516,MTS,129196,4.21,61.05
10,6182512,MTS,107878,3.51,64.57


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 06:
-- Customer Order Profile by Planning-Horizon Status
--
-- Business Question:
-- How are customer orders distributed relative to the
-- planning horizon, and what quantity and sales value
-- are associated with each horizon status?
--
-- SQL Skills:
-- COUNT, COUNT DISTINCT, SUM, ROUND,
-- GROUP BY, CASE, ORDER BY

SELECT
    planning_horizon_status,

    COUNT(*) AS order_count,

    COUNT(DISTINCT product) AS distinct_products,

    SUM(quantity_m) AS total_order_quantity_m,

    ROUND(
        AVG(quantity_m),
        2
    ) AS avg_order_quantity_m,

    ROUND(
        SUM(sales_value_eur),
        2
    ) AS total_sales_value_eur,

    ROUND(
        AVG(sales_value_eur),
        2
    ) AS avg_sales_value_per_order_eur

FROM workspace.gold_sop.fact_customer_order

GROUP BY
    planning_horizon_status

ORDER BY
    CASE planning_horizon_status
        WHEN 'Before horizon start' THEN 1
        WHEN 'Within horizon' THEN 2
        WHEN 'Beyond horizon end' THEN 3
        ELSE 4
    END;

planning_horizon_status,order_count,distinct_products,total_order_quantity_m,avg_order_quantity_m,total_sales_value_eur,avg_sales_value_per_order_eur
Before horizon start,9,9,27652,3072.44,65079.72,7231.08
Within horizon,52,37,886548,17049.0,1918812.66,36900.24


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 07:
-- Highest-Value Customer Orders
--
-- Business Question:
-- Which customer orders carry the highest sales value,
-- and what products, quantities, and due-date positions
-- are associated with them?
--
-- SQL Skills:
-- JOIN, calculated metric, ROUND,
-- ORDER BY, LIMIT

SELECT
    co.order_id,
    co.product,
    p.mts_mto,

    co.due_date_days_in_the_planning_horizon AS due_date_day,

    co.planning_horizon_status,

    co.quantity_m,

    ROUND(
        co.sales_value_eur,
        2
    ) AS sales_value_eur,

    ROUND(
        co.sales_value_eur / NULLIF(co.quantity_m, 0),
        2
    ) AS sales_value_per_meter_eur

FROM workspace.gold_sop.fact_customer_order AS co

LEFT JOIN workspace.gold_sop.dim_product AS p
    ON co.product = p.product

ORDER BY
    co.sales_value_eur DESC

LIMIT 10;

order_id,product,mts_mto,due_date_day,planning_horizon_status,quantity_m,sales_value_eur,sales_value_per_meter_eur
62,7225302,MTO,44,Within horizon,41900,378240.95,9.03
61,7225301,MTO,44,Within horizon,53100,223968.38,4.22
63,7225537,MTO,44,Within horizon,34600,128919.6,3.73
39,6225584,MTO,28,Within horizon,50300,118798.54,2.36
22,6209061,MTS,72,Within horizon,68250,104356.99,1.53
33,6225406,MTO,28,Within horizon,30950,87340.9,2.82
23,6209061,MTS,101,Within horizon,54250,82950.43,1.53
24,6209063,MTO,35,Within horizon,21080,62834.44,2.98
16,6209061,MTS,7,Within horizon,39130,59831.35,1.53
42,7207030,MTS,101,Within horizon,80000,56567.05,0.71


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 08:
-- Initial Inventory vs Planned Demand
--
-- Business Question:
-- For products with planned demand, how does initial inventory
-- compare with total planned demand across the represented
-- demand-planning periods?
--
-- Important:
-- This is NOT a shortage calculation.
-- Production, replenishment, and other supply flows are not
-- included in this comparison.
--
-- SQL Skills:
-- CTE, JOIN, SUM, COALESCE, ROUND,
-- CASE, calculated ratios, ORDER BY

WITH product_demand AS (

    SELECT
        product,
        SUM(adjusted_total_demand_m) AS total_planned_demand_m

    FROM workspace.gold_sop.fact_demand_plan

    GROUP BY product
)

SELECT
    pd.product,
    p.mts_mto,

    p.initial_inventory_m,

    pd.total_planned_demand_m,

    ROUND(
        p.initial_inventory_m / NULLIF(pd.total_planned_demand_m, 0),
        2
    ) AS initial_inventory_to_demand_ratio,

    CASE
        WHEN p.initial_inventory_m >= pd.total_planned_demand_m
            THEN 'Initial inventory >= planned demand'
        ELSE 'Initial inventory < planned demand'
    END AS inventory_demand_comparison

FROM product_demand AS pd

LEFT JOIN workspace.gold_sop.dim_product AS p
    ON pd.product = p.product

ORDER BY
    initial_inventory_to_demand_ratio ASC,
    pd.total_planned_demand_m DESC;

product,mts_mto,initial_inventory_m,total_planned_demand_m,initial_inventory_to_demand_ratio,inventory_demand_comparison
6182514,MTS,286,178270,0.0,Initial inventory < planned demand
6201036,MTS,45,130884,0.0,Initial inventory < planned demand
6182506,MTS,70,69243,0.0,Initial inventory < planned demand
7222012,MTS,0,40300,0.0,Initial inventory < planned demand
7202031,MTS,0,25413,0.0,Initial inventory < planned demand
6225077,MTS,0,17538,0.0,Initial inventory < planned demand
6182513,MTS,1927,243147,0.01,Initial inventory < planned demand
6225111,MTS,1264,168256,0.01,Initial inventory < planned demand
6182512,MTS,1109,107878,0.01,Initial inventory < planned demand
72220021,MTS,200,24480,0.01,Initial inventory < planned demand


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 09:
-- Demand Product Production-Alternative Coverage
--
-- Business Question:
-- Do products with planned demand have defined production
-- alternatives, and how many alternatives are available
-- for each demand-bearing product?
--
-- Important:
-- main_alternative codes 1/2 are preserved exactly as
-- provided by the source. Their business meaning is not inferred.
--
-- SQL Skills:
-- CTE, LEFT JOIN, COUNT DISTINCT,
-- CASE, GROUP BY, ORDER BY

WITH demand_products AS (

    SELECT DISTINCT
        product

    FROM workspace.gold_sop.fact_demand_plan
)

SELECT
    dp.product,

    COUNT(DISTINCT bpp.alternative) AS production_alternative_count,

    COUNT(DISTINCT bam.machine) AS associated_machine_count,

    CASE
        WHEN COUNT(DISTINCT bpp.alternative) = 0
            THEN 'No production alternative'
        WHEN COUNT(DISTINCT bam.machine) = 0
            THEN 'Alternative exists - no machine relationship'
        ELSE 'Production relationship available'
    END AS production_relationship_status

FROM demand_products AS dp

LEFT JOIN workspace.gold_sop.bridge_product_production AS bpp
    ON dp.product = bpp.product

LEFT JOIN workspace.gold_sop.bridge_alternative_machine AS bam
    ON bpp.alternative = bam.alternative

GROUP BY
    dp.product

ORDER BY
    production_alternative_count ASC,
    associated_machine_count ASC,
    dp.product;

product,production_alternative_count,associated_machine_count,production_relationship_status
7202031,0,0,No production alternative
6182506,1,1,Production relationship available
7207043,1,1,Production relationship available
7226005,1,1,Production relationship available
6209061,1,2,Production relationship available
6182512,2,2,Production relationship available
6182513,2,2,Production relationship available
6182514,2,2,Production relationship available
6182521,2,2,Production relationship available
6225077,2,2,Production relationship available


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 10:
-- Machine Capacity by Planning Period
--
-- Business Question:
-- How much regular and overtime production-machine
-- capacity is available across the planning horizon?
--
-- Important:
-- These measures represent AVAILABLE capacity.
-- They do not represent actual machine usage or utilization.
--
-- SQL Skills:
-- JOIN, COUNT DISTINCT, SUM, ROUND,
-- calculated percentages, GROUP BY, ORDER BY

SELECT
    mc.period,
    pp.macroperiod_month,
    pp.number_of_days,

    COUNT(DISTINCT mc.machine) AS machines_with_capacity,

    ROUND(
        SUM(mc.available_regular_time_h),
        2
    ) AS regular_capacity_h,

    ROUND(
        SUM(mc.available_overtime_h),
        2
    ) AS overtime_capacity_h,

    ROUND(
        SUM(mc.available_regular_time_h)
        + SUM(mc.available_overtime_h),
        2
    ) AS total_available_capacity_h,

    ROUND(
        100.0 * SUM(mc.available_overtime_h)
        / NULLIF(
            SUM(mc.available_regular_time_h)
            + SUM(mc.available_overtime_h),
            0
        ),
        2
    ) AS overtime_share_of_capacity_pct

FROM workspace.gold_sop.fact_machine_capacity AS mc

INNER JOIN workspace.gold_sop.dim_planning_period AS pp
    ON mc.period = pp.period

GROUP BY
    mc.period,
    pp.macroperiod_month,
    pp.number_of_days

ORDER BY
    mc.period;

period,macroperiod_month,number_of_days,machines_with_capacity,regular_capacity_h,overtime_capacity_h,total_available_capacity_h,overtime_share_of_capacity_pct
1,1,7,39,1998.4,603.2,2601.6,23.19
2,1,8,39,2304.0,307.2,2611.2,11.76
3,2,6,39,1558.4,304.0,1862.4,16.32
4,2,7,39,1998.4,603.2,2601.6,23.19
5,2,7,39,1998.4,603.2,2601.6,23.19
6,2,9,39,1625.6,601.6,2227.2,27.01
7,3,28,39,7993.6,2412.8,10406.4,23.19
8,4,29,39,7755.2,3006.4,10761.6,27.94


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 11:
-- Machine Capacity Intensity by Planning Period
--
-- Business Question:
-- How much regular and overtime machine capacity is
-- available per planning day?
--
-- Purpose:
-- Normalize capacity for differences in planning-period length.
--
-- SQL Skills:
-- JOIN, SUM, ROUND, arithmetic calculations,
-- GROUP BY, ORDER BY

SELECT
    mc.period,
    pp.macroperiod_month,
    pp.number_of_days,

    COUNT(DISTINCT mc.machine) AS machines_with_capacity,

    ROUND(
        SUM(mc.available_regular_time_h) / pp.number_of_days,
        2
    ) AS regular_capacity_per_day_h,

    ROUND(
        SUM(mc.available_overtime_h) / pp.number_of_days,
        2
    ) AS overtime_capacity_per_day_h,

    ROUND(
        (
            SUM(mc.available_regular_time_h)
            + SUM(mc.available_overtime_h)
        ) / pp.number_of_days,
        2
    ) AS total_capacity_per_day_h

FROM workspace.gold_sop.fact_machine_capacity AS mc

INNER JOIN workspace.gold_sop.dim_planning_period AS pp
    ON mc.period = pp.period

GROUP BY
    mc.period,
    pp.macroperiod_month,
    pp.number_of_days

ORDER BY
    mc.period;

period,macroperiod_month,number_of_days,machines_with_capacity,regular_capacity_per_day_h,overtime_capacity_per_day_h,total_capacity_per_day_h
1,1,7,39,285.49,86.17,371.66
2,1,8,39,288.0,38.4,326.4
3,2,6,39,259.73,50.67,310.4
4,2,7,39,285.49,86.17,371.66
5,2,7,39,285.49,86.17,371.66
6,2,9,39,180.62,66.84,247.47
7,3,28,39,285.49,86.17,371.66
8,4,29,39,267.42,103.67,371.09


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 12:
-- Production Alternative Code Profile
--
-- Business Question:
-- How are the source-provided main_alternative codes
-- distributed across products and production alternatives?
--
-- Purpose:
-- Understand the structure of codes 1/2 before using
-- production alternatives in capacity-requirement calculations.
--
-- Important:
-- No business meaning is assigned to code 1 or code 2.
--
-- SQL Skills:
-- GROUP BY, COUNT, COUNT DISTINCT,
-- conditional aggregation, ORDER BY

SELECT
    main_alternative,

    COUNT(*) AS product_alternative_relationships,

    COUNT(DISTINCT product) AS distinct_products,

    COUNT(DISTINCT alternative) AS distinct_alternatives

FROM workspace.gold_sop.bridge_product_production

GROUP BY
    main_alternative

ORDER BY
    main_alternative;

main_alternative,product_alternative_relationships,distinct_products,distinct_alternatives
1,291,291,291
2,301,198,301


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 12A:
-- Production Alternative Pattern by Product
--
-- Business Question:
-- How many code-1 and code-2 production-alternative
-- relationships exist per product?
--
-- Purpose:
-- Validate the structural pattern of main_alternative
-- before using alternatives in capacity calculations.
--
-- Important:
-- Codes 1 and 2 remain uninterpreted source codes.
--
-- SQL Skills:
-- CTE, CASE, conditional aggregation,
-- COUNT, MIN, MAX, GROUP BY

WITH product_alternative_profile AS (

    SELECT
        product,

        SUM(
            CASE WHEN main_alternative = 1
                 THEN 1 ELSE 0 END
        ) AS code_1_count,

        SUM(
            CASE WHEN main_alternative = 2
                 THEN 1 ELSE 0 END
        ) AS code_2_count

    FROM workspace.gold_sop.bridge_product_production

    GROUP BY product
)

SELECT
    COUNT(*) AS total_products,

    SUM(
        CASE WHEN code_1_count = 1
             THEN 1 ELSE 0 END
    ) AS products_with_exactly_one_code_1,

    SUM(
        CASE WHEN code_1_count = 0
             THEN 1 ELSE 0 END
    ) AS products_with_no_code_1,

    SUM(
        CASE WHEN code_1_count > 1
             THEN 1 ELSE 0 END
    ) AS products_with_multiple_code_1,

    SUM(
        CASE WHEN code_2_count > 0
             THEN 1 ELSE 0 END
    ) AS products_with_code_2,

    MIN(code_2_count) AS min_code_2_count,

    MAX(code_2_count) AS max_code_2_count

FROM product_alternative_profile;

total_products,products_with_exactly_one_code_1,products_with_no_code_1,products_with_multiple_code_1,products_with_code_2,min_code_2_count,max_code_2_count
291,291,0,0,198,0,3


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 13:
-- Code-1 Alternative Machine-Mapping Validation
--
-- Business Question:
-- How many machines are associated with each product's
-- single code-1 production alternative?
--
-- Purpose:
-- Determine whether code-1 relationships provide a unique
-- product-to-machine path that can safely support
-- production-hour calculations.
--
-- Important:
-- Code 1 remains a source code; no unsupported
-- business label is assigned.
--
-- SQL Skills:
-- CTE, LEFT JOIN, COUNT DISTINCT,
-- conditional aggregation, MIN, MAX

WITH code_1_machine_profile AS (

    SELECT
        bpp.product,
        bpp.alternative,

        COUNT(
            DISTINCT bam.machine
        ) AS machine_count

    FROM workspace.gold_sop.bridge_product_production AS bpp

    LEFT JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1

    GROUP BY
        bpp.product,
        bpp.alternative
)

SELECT
    COUNT(*) AS code_1_product_alternatives,

    SUM(
        CASE WHEN machine_count = 0
             THEN 1 ELSE 0 END
    ) AS alternatives_with_no_machine,

    SUM(
        CASE WHEN machine_count = 1
             THEN 1 ELSE 0 END
    ) AS alternatives_with_one_machine,

    SUM(
        CASE WHEN machine_count > 1
             THEN 1 ELSE 0 END
    ) AS alternatives_with_multiple_machines,

    MIN(machine_count) AS min_machine_count,

    MAX(machine_count) AS max_machine_count

FROM code_1_machine_profile;

code_1_product_alternatives,alternatives_with_no_machine,alternatives_with_one_machine,alternatives_with_multiple_machines,min_machine_count,max_machine_count
291,9,248,34,0,4


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 13A:
-- Multi-Machine Alternative Production-Time Profile
--
-- Business Question:
-- When a code-1 production alternative is associated
-- with multiple machines, do those machines have the
-- same or different production times?
--
-- Purpose:
-- Determine whether production requirements can be
-- calculated at the alternative level without
-- double-counting product demand.
--
-- SQL Skills:
-- CTE, JOIN, COUNT DISTINCT,
-- MIN, MAX, GROUP BY, CASE

WITH code_1_multi_machine AS (

    SELECT
        bpp.product,
        bpp.alternative

    FROM workspace.gold_sop.bridge_product_production AS bpp

    INNER JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1

    GROUP BY
        bpp.product,
        bpp.alternative

    HAVING COUNT(DISTINCT bam.machine) > 1
),

production_time_profile AS (

    SELECT
        c.product,
        c.alternative,

        COUNT(DISTINCT bam.machine) AS machine_count,

        COUNT(
            DISTINCT bam.production_time_sec_m
        ) AS distinct_production_times,

        MIN(
            bam.production_time_sec_m
        ) AS min_production_time_sec_m,

        MAX(
            bam.production_time_sec_m
        ) AS max_production_time_sec_m

    FROM code_1_multi_machine AS c

    INNER JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON c.alternative = bam.alternative

    GROUP BY
        c.product,
        c.alternative
)

SELECT
    COUNT(*) AS multi_machine_alternatives,

    SUM(
        CASE
            WHEN distinct_production_times = 1
            THEN 1 ELSE 0
        END
    ) AS same_production_time_across_machines,

    SUM(
        CASE
            WHEN distinct_production_times > 1
            THEN 1 ELSE 0
        END
    ) AS different_production_times_across_machines,

    MIN(machine_count) AS min_machines_per_alternative,

    MAX(machine_count) AS max_machines_per_alternative

FROM production_time_profile;

multi_machine_alternatives,same_production_time_across_machines,different_production_times_across_machines,min_machines_per_alternative,max_machines_per_alternative
34,0,34,2,4


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 13B:
-- Demand Exposure by Production-Mapping Structure
--
-- Business Question:
-- How much planned demand is associated with products
-- having single-machine, multi-machine, or unavailable
-- code-1 production mappings?
--
-- Important:
-- Code 1 remains an uninterpreted source code.
-- This query does not assume an actual machine allocation.
--
-- SQL Skills:
-- Multiple CTEs, LEFT JOIN, CASE,
-- COUNT DISTINCT, SUM, ROUND, window function

WITH product_demand AS (

    SELECT
        product,
        SUM(adjusted_total_demand_m) AS total_planned_demand_m

    FROM workspace.gold_sop.fact_demand_plan

    GROUP BY product
),

code_1_profile AS (

    SELECT
        bpp.product,
        COUNT(DISTINCT bam.machine) AS machine_count

    FROM workspace.gold_sop.bridge_product_production AS bpp

    LEFT JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1

    GROUP BY
        bpp.product
),

classified_demand AS (

    SELECT
        pd.product,
        pd.total_planned_demand_m,

        CASE
            WHEN c1.product IS NULL
                THEN 'No code-1 production alternative'

            WHEN c1.machine_count = 0
                THEN 'Code-1 alternative - no machine'

            WHEN c1.machine_count = 1
                THEN 'Code-1 alternative - single machine'

            ELSE 'Code-1 alternative - multiple machines'
        END AS production_mapping_status

    FROM product_demand AS pd

    LEFT JOIN code_1_profile AS c1
        ON pd.product = c1.product
),

summary AS (

    SELECT
        production_mapping_status,

        COUNT(DISTINCT product) AS product_count,

        SUM(total_planned_demand_m) AS total_planned_demand_m

    FROM classified_demand

    GROUP BY
        production_mapping_status
)

SELECT
    production_mapping_status,
    product_count,
    total_planned_demand_m,

    ROUND(
        100.0 * total_planned_demand_m
        / SUM(total_planned_demand_m) OVER (),
        2
    ) AS planned_demand_share_pct

FROM summary

ORDER BY
    total_planned_demand_m DESC;

production_mapping_status,product_count,total_planned_demand_m,planned_demand_share_pct
Code-1 alternative - multiple machines,24,1583030,51.54
Code-1 alternative - single machine,16,1463249,47.64
No code-1 production alternative,1,25413,0.83


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 14:
-- Production-Hour Requirement Range
--
-- Business Question:
-- Based on code-1 production mappings, what range of
-- processing hours is implied by planned demand?
--
-- Method:
-- Minimum hours = demand × fastest mapped production time
-- Maximum hours = demand × slowest mapped production time
--
-- Important:
-- These are theoretical processing-time scenarios,
-- NOT an actual production schedule or machine allocation.
--
-- Products without a code-1 machine mapping are excluded
-- from the hour calculation and remain separately documented.
--
-- SQL Skills:
-- Multiple CTEs, JOIN, MIN, MAX, SUM,
-- CASE, unit conversion, ROUND

WITH product_demand AS (

    SELECT
        product,
        SUM(adjusted_total_demand_m) AS total_planned_demand_m

    FROM workspace.gold_sop.fact_demand_plan

    GROUP BY product
),

code_1_processing_range AS (

    SELECT
        bpp.product,

        COUNT(DISTINCT bam.machine) AS mapped_machine_count,

        MIN(bam.production_time_sec_m)
            AS fastest_production_time_sec_m,

        MAX(bam.production_time_sec_m)
            AS slowest_production_time_sec_m

    FROM workspace.gold_sop.bridge_product_production AS bpp

    LEFT JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1

    GROUP BY
        bpp.product
)

SELECT
    pd.product,

    pd.total_planned_demand_m,

    c1.mapped_machine_count,

    c1.fastest_production_time_sec_m,

    c1.slowest_production_time_sec_m,

    ROUND(
        pd.total_planned_demand_m
        * c1.fastest_production_time_sec_m
        / 3600.0,
        2
    ) AS minimum_theoretical_processing_h,

    ROUND(
        pd.total_planned_demand_m
        * c1.slowest_production_time_sec_m
        / 3600.0,
        2
    ) AS maximum_theoretical_processing_h,

    CASE
        WHEN c1.product IS NULL
            THEN 'No code-1 production alternative'

        WHEN c1.mapped_machine_count = 0
            THEN 'Code-1 alternative - no machine'

        WHEN c1.mapped_machine_count = 1
            THEN 'Single-machine requirement'

        ELSE 'Multi-machine scenario range'
    END AS requirement_status

FROM product_demand AS pd

LEFT JOIN code_1_processing_range AS c1
    ON pd.product = c1.product

ORDER BY
    maximum_theoretical_processing_h DESC NULLS LAST;

product,total_planned_demand_m,mapped_machine_count,fastest_production_time_sec_m,slowest_production_time_sec_m,minimum_theoretical_processing_h,maximum_theoretical_processing_h,requirement_status
6209061,282207,2,7.4,14.3,580.09,1120.99,Multi-machine scenario range
7222002,350530,2,2.1,10.3,204.48,1002.91,Multi-machine scenario range
7222001,150076,2,2.5,10.3,104.22,429.38,Multi-machine scenario range
6201036,130884,2,4.9,11.8,178.15,429.01,Multi-machine scenario range
6225112,242800,1,3.3,3.3,222.57,222.57,Single-machine requirement
7222008,86784,2,4.6,9.0,110.89,216.96,Multi-machine scenario range
6182513,243147,1,2.6,2.6,175.61,175.61,Single-machine requirement
7222005,46036,3,3.6,13.2,46.04,168.8,Multi-machine scenario range
6201035,31410,2,4.8,14.4,41.88,125.64,Multi-machine scenario range
6225111,168256,1,2.6,2.6,121.52,121.52,Single-machine requirement


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 15:
-- Theoretical Processing-Hour Range by Planning Period
--
-- Business Question:
-- For each planning period, what theoretical production
-- processing-hour range is implied by planned demand?
--
-- Method:
-- Minimum hours = period demand × fastest mapped
-- production time for the product's code-1 alternative.
--
-- Maximum hours = period demand × slowest mapped
-- production time for the product's code-1 alternative.
--
-- Important:
-- These are theoretical processing-time scenarios.
-- They are NOT actual machine utilization or a
-- production schedule.
--
-- Demand without a usable code-1 machine mapping is
-- reported separately rather than assigned zero hours.
--
-- SQL Skills:
-- Multiple CTEs, LEFT JOIN, conditional aggregation,
-- MIN, MAX, SUM, ROUND, unit conversion

WITH code_1_processing_range AS (

    SELECT
        bpp.product,

        COUNT(DISTINCT bam.machine) AS mapped_machine_count,

        MIN(bam.production_time_sec_m)
            AS fastest_production_time_sec_m,

        MAX(bam.production_time_sec_m)
            AS slowest_production_time_sec_m

    FROM workspace.gold_sop.bridge_product_production AS bpp

    LEFT JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1

    GROUP BY
        bpp.product
),

period_requirement AS (

    SELECT
        d.period,

        SUM(d.adjusted_total_demand_m)
            AS total_planned_demand_m,

        SUM(
            CASE
                WHEN c1.mapped_machine_count > 0
                THEN d.adjusted_total_demand_m
                ELSE 0
            END
        ) AS mapped_demand_m,

        SUM(
            CASE
                WHEN c1.product IS NULL
                     OR c1.mapped_machine_count = 0
                THEN d.adjusted_total_demand_m
                ELSE 0
            END
        ) AS unmapped_demand_m,

        SUM(
            CASE
                WHEN c1.mapped_machine_count > 0
                THEN
                    d.adjusted_total_demand_m
                    * c1.fastest_production_time_sec_m
                    / 3600.0
            END
        ) AS minimum_theoretical_processing_h,

        SUM(
            CASE
                WHEN c1.mapped_machine_count > 0
                THEN
                    d.adjusted_total_demand_m
                    * c1.slowest_production_time_sec_m
                    / 3600.0
            END
        ) AS maximum_theoretical_processing_h

    FROM workspace.gold_sop.fact_demand_plan AS d

    LEFT JOIN code_1_processing_range AS c1
        ON d.product = c1.product

    GROUP BY
        d.period
)

SELECT
    p.period,
    p.macroperiod_month,
    p.number_of_days,

    COALESCE(r.total_planned_demand_m, 0)
        AS total_planned_demand_m,

    COALESCE(r.mapped_demand_m, 0)
        AS mapped_demand_m,

    COALESCE(r.unmapped_demand_m, 0)
        AS unmapped_demand_m,

    ROUND(
        COALESCE(r.minimum_theoretical_processing_h, 0),
        2
    ) AS minimum_theoretical_processing_h,

    ROUND(
        COALESCE(r.maximum_theoretical_processing_h, 0),
        2
    ) AS maximum_theoretical_processing_h

FROM workspace.gold_sop.dim_planning_period AS p

LEFT JOIN period_requirement AS r
    ON p.period = r.period

ORDER BY
    p.period;

period,macroperiod_month,number_of_days,total_planned_demand_m,mapped_demand_m,unmapped_demand_m,minimum_theoretical_processing_h,maximum_theoretical_processing_h
1,1,7,0,0,0,0.0,0.0
2,1,8,0,0,0,0.0,0.0
3,2,6,175527,174075,1452,178.6,329.45
4,2,7,204783,203089,1694,208.37,384.36
5,2,7,204783,203089,1694,208.37,384.36
6,2,9,263293,261115,2178,267.91,494.18
7,3,28,1092148,1083112,9036,1111.3,2049.88
8,4,29,1131158,1121799,9359,1150.99,2123.1


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 16:
-- Theoretical Processing Load vs Available Machine Capacity
--
-- Business Question:
-- At the aggregate planning-period level, how does the
-- theoretical processing-hour requirement compare with
-- available machine capacity?
--
-- Important:
-- This is NOT actual machine utilization.
-- No actual machine allocation or production schedule
-- is available in the source.
--
-- Minimum requirement uses the fastest mapped processing time.
-- Maximum requirement uses the slowest mapped processing time.
--
-- Unmapped demand remains separately reported.
--
-- SQL Skills:
-- Multiple CTEs, LEFT JOIN, conditional aggregation,
-- SUM, ROUND, CASE, unit conversion

WITH code_1_processing_range AS (

    SELECT
        bpp.product,

        COUNT(DISTINCT bam.machine) AS mapped_machine_count,

        MIN(bam.production_time_sec_m)
            AS fastest_production_time_sec_m,

        MAX(bam.production_time_sec_m)
            AS slowest_production_time_sec_m

    FROM workspace.gold_sop.bridge_product_production AS bpp

    LEFT JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1

    GROUP BY
        bpp.product
),

period_requirement AS (

    SELECT
        d.period,

        SUM(d.adjusted_total_demand_m)
            AS total_planned_demand_m,

        SUM(
            CASE
                WHEN c1.product IS NULL
                     OR c1.mapped_machine_count = 0
                THEN d.adjusted_total_demand_m
                ELSE 0
            END
        ) AS unmapped_demand_m,

        SUM(
            CASE
                WHEN c1.mapped_machine_count > 0
                THEN
                    d.adjusted_total_demand_m
                    * c1.fastest_production_time_sec_m
                    / 3600.0
            END
        ) AS minimum_required_h,

        SUM(
            CASE
                WHEN c1.mapped_machine_count > 0
                THEN
                    d.adjusted_total_demand_m
                    * c1.slowest_production_time_sec_m
                    / 3600.0
            END
        ) AS maximum_required_h

    FROM workspace.gold_sop.fact_demand_plan AS d

    LEFT JOIN code_1_processing_range AS c1
        ON d.product = c1.product

    GROUP BY
        d.period
),

period_capacity AS (

    SELECT
        period,

        SUM(available_regular_time_h)
            AS regular_capacity_h,

        SUM(available_overtime_h)
            AS overtime_capacity_h,

        SUM(
            available_regular_time_h
            + available_overtime_h
        ) AS total_available_capacity_h

    FROM workspace.gold_sop.fact_machine_capacity

    GROUP BY
        period
)

SELECT
    p.period,
    p.number_of_days,

    COALESCE(r.total_planned_demand_m, 0)
        AS total_planned_demand_m,

    COALESCE(r.unmapped_demand_m, 0)
        AS unmapped_demand_m,

    ROUND(COALESCE(r.minimum_required_h, 0), 2)
        AS minimum_theoretical_processing_h,

    ROUND(COALESCE(r.maximum_required_h, 0), 2)
        AS maximum_theoretical_processing_h,

    ROUND(c.regular_capacity_h, 2)
        AS regular_capacity_h,

    ROUND(c.overtime_capacity_h, 2)
        AS overtime_capacity_h,

    ROUND(c.total_available_capacity_h, 2)
        AS total_available_capacity_h,

    ROUND(
        100.0 * COALESCE(r.minimum_required_h, 0)
        / NULLIF(c.total_available_capacity_h, 0),
        2
    ) AS minimum_load_to_capacity_pct,

    ROUND(
        100.0 * COALESCE(r.maximum_required_h, 0)
        / NULLIF(c.total_available_capacity_h, 0),
        2
    ) AS maximum_load_to_capacity_pct

FROM workspace.gold_sop.dim_planning_period AS p

LEFT JOIN period_requirement AS r
    ON p.period = r.period

LEFT JOIN period_capacity AS c
    ON p.period = c.period

ORDER BY
    p.period;

period,number_of_days,total_planned_demand_m,unmapped_demand_m,minimum_theoretical_processing_h,maximum_theoretical_processing_h,regular_capacity_h,overtime_capacity_h,total_available_capacity_h,minimum_load_to_capacity_pct,maximum_load_to_capacity_pct
1,7,0,0,0.0,0.0,1998.4,603.2,2601.6,0.0,0.0
2,8,0,0,0.0,0.0,2304.0,307.2,2611.2,0.0,0.0
3,6,175527,1452,178.6,329.45,1558.4,304.0,1862.4,9.59,17.69
4,7,204783,1694,208.37,384.36,1998.4,603.2,2601.6,8.01,14.77
5,7,204783,1694,208.37,384.36,1998.4,603.2,2601.6,8.01,14.77
6,9,263293,2178,267.91,494.18,1625.6,601.6,2227.2,12.03,22.19
7,28,1092148,9036,1111.3,2049.88,7993.6,2412.8,10406.4,10.68,19.7
8,29,1131158,9359,1150.99,2123.1,7755.2,3006.4,10761.6,10.7,19.73


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 17:
-- Machine Eligibility Concentration for Demand-Bearing Products
--
-- Business Question:
-- Which machines are eligible to support the code-1
-- production alternatives of products with planned demand?
--
-- Purpose:
-- Identify machines that appear across many demand-bearing
-- product routes before evaluating machine-level capacity.
--
-- Important:
-- This measures production-route eligibility.
-- It does NOT represent actual machine assignment,
-- workload, utilization, or production volume.
--
-- SQL Skills:
-- CTE, DISTINCT, JOIN, COUNT,
-- SUM, ROUND, GROUP BY, ORDER BY

WITH demand_products AS (

    SELECT
        product,
        SUM(adjusted_total_demand_m) AS total_planned_demand_m

    FROM workspace.gold_sop.fact_demand_plan

    GROUP BY
        product
),

eligible_routes AS (

    SELECT DISTINCT
        dp.product,
        dp.total_planned_demand_m,
        bam.machine

    FROM demand_products AS dp

    INNER JOIN workspace.gold_sop.bridge_product_production AS bpp
        ON dp.product = bpp.product

    INNER JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1
)

SELECT
    machine,

    COUNT(DISTINCT product)
        AS eligible_demand_products,

    SUM(total_planned_demand_m)
        AS demand_m_associated_with_eligible_routes,

    ROUND(
        100.0 * SUM(total_planned_demand_m)
        / SUM(SUM(total_planned_demand_m)) OVER (),
        2
    ) AS share_of_route_associated_demand_pct

FROM eligible_routes

GROUP BY
    machine

ORDER BY
    eligible_demand_products DESC,
    demand_m_associated_with_eligible_routes DESC;

machine,eligible_demand_products,demand_m_associated_with_eligible_routes,share_of_route_associated_demand_pct
316,23,1657108,34.83
590,13,630730,13.26
324,10,297519,6.25
327,5,606547,12.75
543,3,525086,11.04
325,3,207051,4.35
555,2,328243,6.90
343,2,162294,3.41
375,2,127084,2.67
377,2,104322,2.19


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 18:
-- Eligible-Machine Capacity Profile
--
-- Business Question:
-- What available capacity exists on machines that are
-- eligible for demand-bearing code-1 production routes?
--
-- Purpose:
-- Combine production-route importance with available
-- machine capacity before developing allocation scenarios.
--
-- Important:
-- This is AVAILABLE capacity, not utilization.
-- Demand has NOT been allocated to machines.
--
-- SQL Skills:
-- Multiple CTEs, DISTINCT, JOIN,
-- COUNT, SUM, ROUND, GROUP BY, ORDER BY

WITH demand_products AS (

    SELECT DISTINCT
        product

    FROM workspace.gold_sop.fact_demand_plan
),

eligible_routes AS (

    SELECT DISTINCT
        dp.product,
        bam.machine

    FROM demand_products AS dp

    INNER JOIN workspace.gold_sop.bridge_product_production AS bpp
        ON dp.product = bpp.product

    INNER JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1
),

machine_route_profile AS (

    SELECT
        machine,
        COUNT(DISTINCT product)
            AS eligible_demand_products

    FROM eligible_routes

    GROUP BY
        machine
),

machine_capacity_profile AS (

    SELECT
        machine,

        SUM(available_regular_time_h)
            AS total_regular_capacity_h,

        SUM(available_overtime_h)
            AS total_overtime_capacity_h,

        SUM(
            available_regular_time_h
            + available_overtime_h
        ) AS total_available_capacity_h

    FROM workspace.gold_sop.fact_machine_capacity

    GROUP BY
        machine
)

SELECT
    r.machine,

    r.eligible_demand_products,

    ROUND(c.total_regular_capacity_h, 2)
        AS total_regular_capacity_h,

    ROUND(c.total_overtime_capacity_h, 2)
        AS total_overtime_capacity_h,

    ROUND(c.total_available_capacity_h, 2)
        AS total_available_capacity_h,

    ROUND(
        100.0 * c.total_overtime_capacity_h
        / NULLIF(c.total_available_capacity_h, 0),
        2
    ) AS overtime_share_of_available_capacity_pct

FROM machine_route_profile AS r

LEFT JOIN machine_capacity_profile AS c
    ON r.machine = c.machine

ORDER BY
    r.eligible_demand_products DESC,
    c.total_available_capacity_h DESC;

machine,eligible_demand_products,total_regular_capacity_h,total_overtime_capacity_h,total_available_capacity_h,overtime_share_of_available_capacity_pct
316,23,652.8,268.8,921.6,29.17
590,13,921.6,0.0,921.6,0.0
324,10,652.8,268.8,921.6,29.17
327,5,652.8,268.8,921.6,29.17
325,3,652.8,268.8,921.6,29.17
543,3,652.8,268.8,921.6,29.17
375,2,652.8,268.8,921.6,29.17
343,2,652.8,268.8,921.6,29.17
319,2,652.8,268.8,921.6,29.17
555,2,652.8,268.8,921.6,29.17


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 19:
-- Eligible-Machine Capacity by Planning Period
--
-- Business Question:
-- How does available capacity vary by planning period
-- for machines eligible to support demand-bearing
-- code-1 production routes?
--
-- Purpose:
-- Move from horizon-level capacity to period-level
-- resource availability before any workload-allocation
-- analysis.
--
-- Important:
-- These are AVAILABLE capacity hours.
-- They do NOT represent actual machine utilization.
--
-- SQL Skills:
-- Multiple CTEs, DISTINCT, JOIN,
-- ROUND, arithmetic, ORDER BY

WITH demand_products AS (

    SELECT DISTINCT
        product

    FROM workspace.gold_sop.fact_demand_plan
),

eligible_machines AS (

    SELECT DISTINCT
        bam.machine

    FROM demand_products AS dp

    INNER JOIN workspace.gold_sop.bridge_product_production AS bpp
        ON dp.product = bpp.product

    INNER JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1
)

SELECT
    c.machine,
    c.period,
    p.number_of_days,

    ROUND(
        c.available_regular_time_h,
        2
    ) AS regular_capacity_h,

    ROUND(
        c.available_overtime_h,
        2
    ) AS overtime_capacity_h,

    ROUND(
        c.available_regular_time_h
        + c.available_overtime_h,
        2
    ) AS total_available_capacity_h,

    ROUND(
        (
            c.available_regular_time_h
            + c.available_overtime_h
        ) / p.number_of_days,
        2
    ) AS available_capacity_h_per_day

FROM workspace.gold_sop.fact_machine_capacity AS c

INNER JOIN eligible_machines AS e
    ON c.machine = e.machine

INNER JOIN workspace.gold_sop.dim_planning_period AS p
    ON c.period = p.period

ORDER BY
    c.machine,
    c.period;

machine,period,number_of_days,regular_capacity_h,overtime_capacity_h,total_available_capacity_h,available_capacity_h_per_day
316,1,7,48.0,19.2,67.2,9.6
316,2,8,57.6,9.6,67.2,8.4
316,3,6,38.4,9.6,48.0,8.0
316,4,7,48.0,19.2,67.2,9.6
316,5,7,48.0,19.2,67.2,9.6
316,6,9,38.4,19.2,57.6,6.4
316,7,28,192.0,76.8,268.8,9.6
316,8,29,182.4,96.0,278.4,9.6
318,1,7,48.0,19.2,67.2,9.6
318,2,8,57.6,9.6,67.2,8.4


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 20:
-- Single-Machine Product Load vs Available Capacity
--
-- Business Question:
-- For demand-bearing products with exactly one eligible
-- code-1 machine, how much theoretical processing load
-- is associated with that machine by planning period?
--
-- Scope:
-- ONLY products whose code-1 alternative maps to
-- exactly one machine.
--
-- Important:
-- Multi-machine products are intentionally excluded.
-- This is theoretical demand-derived processing load,
-- not actual historical machine utilization.
--
-- SQL Skills:
-- Multiple CTEs, HAVING, JOIN, SUM,
-- unit conversion, CASE, ROUND

WITH code_1_machine_profile AS (

    SELECT
        bpp.product,

        COUNT(DISTINCT bam.machine)
            AS machine_count,

        MIN(bam.machine)
            AS machine,

        MIN(bam.production_time_sec_m)
            AS production_time_sec_m

    FROM workspace.gold_sop.bridge_product_production AS bpp

    INNER JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1

    GROUP BY
        bpp.product

    HAVING COUNT(DISTINCT bam.machine) = 1
),

single_machine_load AS (

    SELECT
        p.machine,
        d.period,

        COUNT(DISTINCT d.product)
            AS demand_products,

        SUM(d.adjusted_total_demand_m)
            AS planned_demand_m,

        SUM(
            d.adjusted_total_demand_m
            * p.production_time_sec_m
            / 3600.0
        ) AS theoretical_processing_h

    FROM workspace.gold_sop.fact_demand_plan AS d

    INNER JOIN code_1_machine_profile AS p
        ON d.product = p.product

    GROUP BY
        p.machine,
        d.period
)

SELECT
    l.machine,
    l.period,

    l.demand_products,

    l.planned_demand_m,

    ROUND(
        l.theoretical_processing_h,
        2
    ) AS theoretical_processing_h,

    ROUND(
        c.available_regular_time_h,
        2
    ) AS regular_capacity_h,

    ROUND(
        c.available_overtime_h,
        2
    ) AS overtime_capacity_h,

    ROUND(
        c.available_regular_time_h
        + c.available_overtime_h,
        2
    ) AS total_available_capacity_h,

    ROUND(
        100.0 * l.theoretical_processing_h
        / NULLIF(
            c.available_regular_time_h
            + c.available_overtime_h,
            0
        ),
        2
    ) AS theoretical_load_to_capacity_pct,

    CASE
        WHEN l.theoretical_processing_h
             > c.available_regular_time_h
               + c.available_overtime_h
            THEN 'Above total available capacity'

        WHEN l.theoretical_processing_h
             > c.available_regular_time_h
            THEN 'Above regular capacity; within total capacity'

        ELSE 'Within regular capacity'
    END AS capacity_position

FROM single_machine_load AS l

INNER JOIN workspace.gold_sop.fact_machine_capacity AS c
    ON l.machine = c.machine
   AND l.period = c.period

ORDER BY
    theoretical_load_to_capacity_pct DESC,
    l.machine,
    l.period;

machine,period,demand_products,planned_demand_m,theoretical_processing_h,regular_capacity_h,overtime_capacity_h,total_available_capacity_h,theoretical_load_to_capacity_pct,capacity_position
316,6,8,104297,81.66,38.4,19.2,57.6,141.77,Above total available capacity
316,7,8,432632,338.73,192.0,76.8,268.8,126.01,Above total available capacity
316,8,8,448084,350.83,182.4,96.0,278.4,126.01,Above total available capacity
316,3,8,69532,54.44,38.4,9.6,48.0,113.42,Above total available capacity
316,4,8,81120,63.51,48.0,19.2,67.2,94.51,Above regular capacity; within total capacity
316,5,8,81120,63.51,48.0,19.2,67.2,94.51,Above regular capacity; within total capacity
324,6,3,8294,12.34,38.4,19.2,57.6,21.43,Within regular capacity
324,7,3,34404,51.2,192.0,76.8,268.8,19.05,Within regular capacity
324,8,3,35632,53.02,182.4,96.0,278.4,19.05,Within regular capacity
324,3,3,5529,8.23,38.4,9.6,48.0,17.14,Within regular capacity


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 21:
-- Machine 316 Capacity Gap Analysis — Complete Planning Horizon
--
-- Business Question:
-- Across all planning periods, how does the theoretical
-- processing requirement from uniquely mapped products
-- compare with Machine 316's available capacity?
--
-- Important:
-- All 8 planning periods are retained.
-- Periods without demand display zero theoretical load.
--
-- This is a planning scenario based on demand-derived
-- processing requirements.
-- It is NOT actual historical utilization.
--
-- Positive gap = requirement exceeds capacity.
-- Negative gap = remaining capacity under this scope.
--
-- SQL Skills:
-- Multiple CTEs, LEFT JOIN, COALESCE,
-- SUM, arithmetic, CASE, ROUND, unit conversion

WITH machine_316_products AS (

    SELECT
        bpp.product,

        MIN(bam.production_time_sec_m)
            AS production_time_sec_m

    FROM workspace.gold_sop.bridge_product_production AS bpp

    INNER JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1

    GROUP BY
        bpp.product

    HAVING COUNT(DISTINCT bam.machine) = 1
       AND MIN(bam.machine) = 316
),

period_load AS (

    SELECT
        d.period,

        SUM(
            d.adjusted_total_demand_m
            * p.production_time_sec_m
            / 3600.0
        ) AS theoretical_processing_h

    FROM workspace.gold_sop.fact_demand_plan AS d

    INNER JOIN machine_316_products AS p
        ON d.product = p.product

    GROUP BY
        d.period
)

SELECT
    pp.period,

    pp.number_of_days,

    ROUND(
        COALESCE(l.theoretical_processing_h, 0),
        2
    ) AS theoretical_processing_h,

    ROUND(
        c.available_regular_time_h,
        2
    ) AS regular_capacity_h,

    ROUND(
        c.available_overtime_h,
        2
    ) AS overtime_capacity_h,

    ROUND(
        c.available_regular_time_h
        + c.available_overtime_h,
        2
    ) AS total_available_capacity_h,

    ROUND(
        COALESCE(l.theoretical_processing_h, 0)
        - c.available_regular_time_h,
        2
    ) AS gap_vs_regular_capacity_h,

    ROUND(
        COALESCE(l.theoretical_processing_h, 0)
        - (
            c.available_regular_time_h
            + c.available_overtime_h
        ),
        2
    ) AS gap_vs_total_capacity_h,

    CASE
        WHEN COALESCE(l.theoretical_processing_h, 0)
             > c.available_regular_time_h
               + c.available_overtime_h
            THEN 'Total capacity gap'

        WHEN COALESCE(l.theoretical_processing_h, 0)
             > c.available_regular_time_h
            THEN 'Overtime required within available envelope'

        ELSE 'Regular capacity sufficient'
    END AS capacity_position

FROM workspace.gold_sop.dim_planning_period AS pp

INNER JOIN workspace.gold_sop.fact_machine_capacity AS c
    ON pp.period = c.period
   AND c.machine = 316

LEFT JOIN period_load AS l
    ON pp.period = l.period

ORDER BY
    pp.period;

period,number_of_days,theoretical_processing_h,regular_capacity_h,overtime_capacity_h,total_available_capacity_h,gap_vs_regular_capacity_h,gap_vs_total_capacity_h,capacity_position
1,7,0.0,48.0,19.2,67.2,-48.0,-67.2,Regular capacity sufficient
2,8,0.0,57.6,9.6,67.2,-57.6,-67.2,Regular capacity sufficient
3,6,54.44,38.4,9.6,48.0,16.04,6.44,Total capacity gap
4,7,63.51,48.0,19.2,67.2,15.51,-3.69,Overtime required within available envelope
5,7,63.51,48.0,19.2,67.2,15.51,-3.69,Overtime required within available envelope
6,9,81.66,38.4,19.2,57.6,43.26,24.06,Total capacity gap
7,28,338.73,192.0,76.8,268.8,146.73,69.93,Total capacity gap
8,29,350.83,182.4,96.0,278.4,168.43,72.43,Total capacity gap


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 22:
-- Machine 316 Product Load Drivers
--
-- Business Question:
-- Which uniquely mapped products contribute the most
-- theoretical processing hours to Machine 316?
--
-- Scope:
-- Demand-bearing products whose code-1 alternative
-- maps to exactly one machine: Machine 316.
--
-- Important:
-- Processing hours are demand-derived theoretical
-- requirements, not actual production hours.
--
-- SQL Skills:
-- Multiple CTEs, HAVING, JOIN,
-- SUM, ROUND, window functions, ranking

WITH machine_316_products AS (

    SELECT
        bpp.product,

        MIN(bam.production_time_sec_m)
            AS production_time_sec_m

    FROM workspace.gold_sop.bridge_product_production AS bpp

    INNER JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1

    GROUP BY
        bpp.product

    HAVING COUNT(DISTINCT bam.machine) = 1
       AND MIN(bam.machine) = 316
),

product_load AS (

    SELECT
        d.product,

        SUM(d.adjusted_total_demand_m)
            AS total_planned_demand_m,

        p.production_time_sec_m,

        SUM(
            d.adjusted_total_demand_m
            * p.production_time_sec_m
            / 3600.0
        ) AS theoretical_processing_h

    FROM workspace.gold_sop.fact_demand_plan AS d

    INNER JOIN machine_316_products AS p
        ON d.product = p.product

    GROUP BY
        d.product,
        p.production_time_sec_m
),

ranked_load AS (

    SELECT
        *,

        RANK() OVER (
            ORDER BY theoretical_processing_h DESC
        ) AS processing_load_rank,

        100.0 * theoretical_processing_h
        / SUM(theoretical_processing_h) OVER ()
            AS share_of_machine_316_theoretical_load_pct

    FROM product_load
)

SELECT
    processing_load_rank,
    product,
    total_planned_demand_m,
    production_time_sec_m,

    ROUND(
        theoretical_processing_h,
        2
    ) AS theoretical_processing_h,

    ROUND(
        share_of_machine_316_theoretical_load_pct,
        2
    ) AS share_of_machine_316_theoretical_load_pct

FROM ranked_load

ORDER BY
    processing_load_rank,
    product;

processing_load_rank,product,total_planned_demand_m,production_time_sec_m,theoretical_processing_h,share_of_machine_316_theoretical_load_pct
1,6225112,242800,3.3,222.57,23.36
2,6182513,243147,2.6,175.61,18.43
3,6225111,168256,2.6,121.52,12.76
4,6182514,178270,2.4,118.85,12.48
5,6182512,107878,3.3,98.89,10.38
6,6182516,129196,2.4,86.13,9.04
7,6182515,92564,2.6,66.85,7.02
8,6182521,54674,4.1,62.27,6.54


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 23:
-- Machine 316 Product Load Drivers by Planning Period
--
-- Business Question:
-- Which uniquely mapped products contribute the most
-- theoretical processing hours to Machine 316
-- in each demand-bearing planning period?
--
-- Important:
-- These are demand-derived theoretical processing
-- requirements, not actual production hours.
--
-- SQL Skills:
-- Multiple CTEs, HAVING, JOIN,
-- window functions, PARTITION BY, ranking

WITH machine_316_products AS (

    SELECT
        bpp.product,

        MIN(bam.production_time_sec_m)
            AS production_time_sec_m

    FROM workspace.gold_sop.bridge_product_production AS bpp

    INNER JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1

    GROUP BY
        bpp.product

    HAVING COUNT(DISTINCT bam.machine) = 1
       AND MIN(bam.machine) = 316
),

period_product_load AS (

    SELECT
        d.period,
        d.product,
        d.adjusted_total_demand_m
            AS planned_demand_m,

        p.production_time_sec_m,

        d.adjusted_total_demand_m
        * p.production_time_sec_m
        / 3600.0
            AS theoretical_processing_h

    FROM workspace.gold_sop.fact_demand_plan AS d

    INNER JOIN machine_316_products AS p
        ON d.product = p.product
),

ranked_load AS (

    SELECT
        *,

        RANK() OVER (
            PARTITION BY period
            ORDER BY theoretical_processing_h DESC
        ) AS load_rank_in_period,

        100.0 * theoretical_processing_h
        / SUM(theoretical_processing_h)
            OVER (PARTITION BY period)
            AS share_of_period_machine_load_pct

    FROM period_product_load
)

SELECT
    period,
    load_rank_in_period,
    product,
    planned_demand_m,
    production_time_sec_m,

    ROUND(
        theoretical_processing_h,
        2
    ) AS theoretical_processing_h,

    ROUND(
        share_of_period_machine_load_pct,
        2
    ) AS share_of_period_machine_load_pct

FROM ranked_load

ORDER BY
    period,
    load_rank_in_period,
    product;

period,load_rank_in_period,product,planned_demand_m,production_time_sec_m,theoretical_processing_h,share_of_period_machine_load_pct
3,1,6225112,13875,3.3,12.72,23.36
3,2,6182513,13894,2.6,10.03,18.43
3,3,6225111,9615,2.6,6.94,12.76
3,4,6182514,10187,2.4,6.79,12.47
3,5,6182512,6165,3.3,5.65,10.38
3,6,6182516,7383,2.4,4.92,9.04
3,7,6182515,5289,2.6,3.82,7.02
3,8,6182521,3124,4.1,3.56,6.54
4,1,6225112,16187,3.3,14.84,23.36
4,2,6182513,16210,2.6,11.71,18.43


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 24:
-- Demand Product Packing-Machine Mapping Structure
--
-- Business Question:
-- How are demand-bearing products mapped to
-- available packing-machine alternatives?
--
-- Purpose:
-- Determine whether packing requirements can be
-- assigned directly or whether routing ambiguity
-- must be handled before capacity analysis.
--
-- Important:
-- Do not assume multiple packing-machine mappings
-- represent actual allocation.
--
-- SQL Skills:
-- Multiple CTEs, LEFT JOIN, COUNT DISTINCT,
-- CASE, GROUP BY

WITH demand_products AS (

    SELECT DISTINCT
        product

    FROM workspace.gold_sop.fact_demand_plan
),

packing_profile AS (

    SELECT
        d.product,

        COUNT(DISTINCT bpp.packing_machine)
            AS packing_machine_count

    FROM demand_products AS d

    LEFT JOIN workspace.gold_sop.bridge_product_packing AS bpp
        ON d.product = bpp.product

    GROUP BY
        d.product
),

classified_products AS (

    SELECT
        product,
        packing_machine_count,

        CASE
            WHEN packing_machine_count = 0
                THEN 'No packing-machine mapping'

            WHEN packing_machine_count = 1
                THEN 'Single packing machine'

            ELSE 'Multiple packing machines'
        END AS packing_mapping_structure

    FROM packing_profile
)

SELECT
    packing_mapping_structure,

    COUNT(*) AS demand_products,

    MIN(packing_machine_count)
        AS min_packing_machines,

    MAX(packing_machine_count)
        AS max_packing_machines

FROM classified_products

GROUP BY
    packing_mapping_structure

ORDER BY
    CASE packing_mapping_structure
        WHEN 'Single packing machine' THEN 1
        WHEN 'Multiple packing machines' THEN 2
        WHEN 'No packing-machine mapping' THEN 3
    END;

packing_mapping_structure,demand_products,min_packing_machines,max_packing_machines
Multiple packing machines,41,2,3


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 25:
-- Packing-Time Variation Across Eligible Machines
--
-- Business Question:
-- For demand-bearing products with multiple eligible
-- packing machines, do packing times vary by machine?
--
-- Purpose:
-- Determine whether packing-machine selection affects
-- theoretical packing-hour requirements.
--
-- Important:
-- Multiple eligible machines do NOT represent actual
-- demand allocation.
--
-- SQL Skills:
-- CTE, JOIN, COUNT DISTINCT,
-- MIN, MAX, CASE, GROUP BY

WITH demand_products AS (

    SELECT DISTINCT
        product

    FROM workspace.gold_sop.fact_demand_plan
),

packing_profile AS (

    SELECT
        d.product,

        COUNT(DISTINCT bpp.packing_machine)
            AS packing_machine_count,

        COUNT(DISTINCT bpp.packing_time_sec_m)
            AS distinct_packing_time_count,

        MIN(bpp.packing_time_sec_m)
            AS min_packing_time_sec_m,

        MAX(bpp.packing_time_sec_m)
            AS max_packing_time_sec_m

    FROM demand_products AS d

    INNER JOIN workspace.gold_sop.bridge_product_packing AS bpp
        ON d.product = bpp.product

    GROUP BY
        d.product
)

SELECT
    COUNT(*) AS demand_products,

    SUM(
        CASE
            WHEN distinct_packing_time_count = 1
                THEN 1
            ELSE 0
        END
    ) AS products_with_same_packing_time,

    SUM(
        CASE
            WHEN distinct_packing_time_count > 1
                THEN 1
            ELSE 0
        END
    ) AS products_with_different_packing_times,

    MIN(packing_machine_count)
        AS min_eligible_packing_machines,

    MAX(packing_machine_count)
        AS max_eligible_packing_machines,

    MIN(min_packing_time_sec_m)
        AS minimum_observed_packing_time_sec_m,

    MAX(max_packing_time_sec_m)
        AS maximum_observed_packing_time_sec_m

FROM packing_profile;

demand_products,products_with_same_packing_time,products_with_different_packing_times,min_eligible_packing_machines,max_eligible_packing_machines,minimum_observed_packing_time_sec_m,maximum_observed_packing_time_sec_m
41,0,41,2,3,0.9,21.7


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 26:
-- Product-Level Theoretical Packing-Hour Range
--
-- Business Question:
-- Given multiple eligible packing machines with
-- different packing times, what is the theoretical
-- minimum-to-maximum packing-hour requirement
-- for each demand-bearing product?
--
-- Interpretation:
-- Minimum hours = fastest eligible packing-machine time
-- Maximum hours = slowest eligible packing-machine time
--
-- These are scenario bounds.
-- They are NOT actual machine assignments or utilization.
--
-- SQL Skills:
-- Multiple CTEs, JOIN, MIN, MAX,
-- COUNT DISTINCT, SUM, unit conversion, ROUND

WITH product_packing_profile AS (

    SELECT
        product,

        COUNT(DISTINCT packing_machine)
            AS eligible_packing_machines,

        MIN(packing_time_sec_m)
            AS fastest_packing_time_sec_m,

        MAX(packing_time_sec_m)
            AS slowest_packing_time_sec_m

    FROM workspace.gold_sop.bridge_product_packing

    GROUP BY
        product
),

product_demand AS (

    SELECT
        product,

        SUM(adjusted_total_demand_m)
            AS total_planned_demand_m

    FROM workspace.gold_sop.fact_demand_plan

    GROUP BY
        product
)

SELECT
    d.product,

    d.total_planned_demand_m,

    p.eligible_packing_machines,

    p.fastest_packing_time_sec_m,

    p.slowest_packing_time_sec_m,

    ROUND(
        d.total_planned_demand_m
        * p.fastest_packing_time_sec_m
        / 3600.0,
        2
    ) AS minimum_theoretical_packing_h,

    ROUND(
        d.total_planned_demand_m
        * p.slowest_packing_time_sec_m
        / 3600.0,
        2
    ) AS maximum_theoretical_packing_h,

    ROUND(
        (
            d.total_planned_demand_m
            * p.slowest_packing_time_sec_m
            / 3600.0
        )
        -
        (
            d.total_planned_demand_m
            * p.fastest_packing_time_sec_m
            / 3600.0
        ),
        2
    ) AS packing_hour_range_h

FROM product_demand AS d

INNER JOIN product_packing_profile AS p
    ON d.product = p.product

ORDER BY
    maximum_theoretical_packing_h DESC,
    d.product;

product,total_planned_demand_m,eligible_packing_machines,fastest_packing_time_sec_m,slowest_packing_time_sec_m,minimum_theoretical_packing_h,maximum_theoretical_packing_h,packing_hour_range_h
6209061,282207,2,2.7,21.7,211.66,1701.08,1489.43
7222002,350530,3,1.5,2.1,146.05,204.48,58.42
6182513,243147,3,1.2,2.1,81.05,141.84,60.79
6225112,242800,3,1.2,2.1,80.93,141.63,60.7
6225111,168256,3,1.2,2.7,56.09,126.19,70.11
6182514,178270,3,1.2,2.1,59.42,103.99,44.57
6201036,130884,3,1.2,2.7,43.63,98.16,54.54
6182516,129196,3,1.2,2.7,43.07,96.9,53.83
7222001,150076,3,1.5,2.1,62.53,87.54,25.01
6182512,107878,3,1.2,2.7,35.96,80.91,44.95


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 27:
-- Packing Route Detail for Product 6209061
--
-- Business Question:
-- Which packing machines are eligible for product 6209061,
-- and what packing time is associated with each route?
--
-- Purpose:
-- Investigate the unusually wide theoretical packing-hour
-- range identified in Step 26.
--
-- Important:
-- Eligibility does not mean actual allocation or usage.
--
-- SQL Skills:
-- JOIN, filtering, ROUND

SELECT
    bpp.product,
    bpp.packing_machine,

    bpp.packing_time_sec_m,

    ROUND(
        bpp.packing_time_sec_m / 60.0,
        4
    ) AS packing_time_min_m,

    ROUND(
        bpp.packing_time_sec_m / 3600.0,
        6
    ) AS packing_time_h_m

FROM workspace.gold_sop.bridge_product_packing AS bpp

WHERE bpp.product = 6209061

ORDER BY
    bpp.packing_time_sec_m,
    bpp.packing_machine;

product,packing_machine,packing_time_sec_m,packing_time_min_m,packing_time_h_m
6209061,924,2.7,0.045,7.5E-4
6209061,927,21.7,0.3617,0.006028


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 28:
-- Packing Capacity by Planning Period
--
-- Business Question:
-- How much regular, overtime, and total packing capacity
-- is available in each planning period?
--
-- Important:
-- These are AVAILABLE capacity hours.
-- They are NOT actual utilized packing hours.
--
-- SQL Skills:
-- JOIN, COUNT DISTINCT, SUM,
-- arithmetic, ROUND, GROUP BY

SELECT
    pp.period,
    pp.macroperiod_month,
    pp.number_of_days,

    COUNT(DISTINCT pc.packing_machine)
        AS packing_machines,

    ROUND(
        SUM(pc.available_regular_time_h),
        2
    ) AS regular_capacity_h,

    ROUND(
        SUM(pc.available_overtime_h),
        2
    ) AS overtime_capacity_h,

    ROUND(
        SUM(
            pc.available_regular_time_h
            + pc.available_overtime_h
        ),
        2
    ) AS total_available_capacity_h,

    ROUND(
        100.0 * SUM(pc.available_overtime_h)
        / NULLIF(
            SUM(
                pc.available_regular_time_h
                + pc.available_overtime_h
            ),
            0
        ),
        2
    ) AS overtime_share_of_capacity_pct

FROM workspace.gold_sop.dim_planning_period AS pp

LEFT JOIN workspace.gold_sop.fact_packing_capacity AS pc
    ON pp.period = pc.period

GROUP BY
    pp.period,
    pp.macroperiod_month,
    pp.number_of_days

ORDER BY
    pp.period;

period,macroperiod_month,number_of_days,packing_machines,regular_capacity_h,overtime_capacity_h,total_available_capacity_h,overtime_share_of_capacity_pct
1,1,7,7,132.0,220.8,352.8,62.59
2,1,8,7,158.4,194.4,352.8,55.1
3,2,6,7,105.6,146.4,252.0,58.1
4,2,7,7,132.0,220.8,352.8,62.59
5,2,7,7,132.0,220.8,352.8,62.59
6,2,9,7,105.6,196.8,302.4,65.08
7,3,28,7,528.0,883.2,1411.2,62.59
8,4,29,7,501.6,960.0,1461.6,65.68


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 29:
-- Theoretical Packing Load Range vs Available Capacity
--
-- Business Question:
-- By planning period, how does the theoretical
-- packing-hour requirement range compare with
-- aggregate available packing capacity?
--
-- Scenario Definitions:
-- Minimum requirement = each product uses its
-- fastest eligible packing time.
--
-- Maximum requirement = each product uses its
-- slowest eligible packing time.
--
-- Important:
-- These are theoretical scenario bounds.
-- They are NOT actual packing-machine assignments
-- or historical utilization.
--
-- SQL Skills:
-- Multiple CTEs, LEFT JOIN, MIN, MAX,
-- SUM, COALESCE, ROUND, NULLIF

WITH product_packing_profile AS (

    SELECT
        product,

        MIN(packing_time_sec_m)
            AS fastest_packing_time_sec_m,

        MAX(packing_time_sec_m)
            AS slowest_packing_time_sec_m

    FROM workspace.gold_sop.bridge_product_packing

    GROUP BY
        product
),

period_packing_load AS (

    SELECT
        d.period,

        SUM(
            d.adjusted_total_demand_m
            * p.fastest_packing_time_sec_m
            / 3600.0
        ) AS minimum_theoretical_packing_h,

        SUM(
            d.adjusted_total_demand_m
            * p.slowest_packing_time_sec_m
            / 3600.0
        ) AS maximum_theoretical_packing_h

    FROM workspace.gold_sop.fact_demand_plan AS d

    INNER JOIN product_packing_profile AS p
        ON d.product = p.product

    GROUP BY
        d.period
),

period_capacity AS (

    SELECT
        period,

        SUM(available_regular_time_h)
            AS regular_capacity_h,

        SUM(available_overtime_h)
            AS overtime_capacity_h,

        SUM(
            available_regular_time_h
            + available_overtime_h
        ) AS total_available_capacity_h

    FROM workspace.gold_sop.fact_packing_capacity

    GROUP BY
        period
)

SELECT
    pp.period,
    pp.number_of_days,

    ROUND(
        COALESCE(l.minimum_theoretical_packing_h, 0),
        2
    ) AS minimum_theoretical_packing_h,

    ROUND(
        COALESCE(l.maximum_theoretical_packing_h, 0),
        2
    ) AS maximum_theoretical_packing_h,

    ROUND(
        c.regular_capacity_h,
        2
    ) AS regular_capacity_h,

    ROUND(
        c.overtime_capacity_h,
        2
    ) AS overtime_capacity_h,

    ROUND(
        c.total_available_capacity_h,
        2
    ) AS total_available_capacity_h,

    ROUND(
        100.0
        * COALESCE(l.minimum_theoretical_packing_h, 0)
        / NULLIF(c.total_available_capacity_h, 0),
        2
    ) AS minimum_load_to_capacity_pct,

    ROUND(
        100.0
        * COALESCE(l.maximum_theoretical_packing_h, 0)
        / NULLIF(c.total_available_capacity_h, 0),
        2
    ) AS maximum_load_to_capacity_pct

FROM workspace.gold_sop.dim_planning_period AS pp

INNER JOIN period_capacity AS c
    ON pp.period = c.period

LEFT JOIN period_packing_load AS l
    ON pp.period = l.period

ORDER BY
    pp.period;

period,number_of_days,minimum_theoretical_packing_h,maximum_theoretical_packing_h,regular_capacity_h,overtime_capacity_h,total_available_capacity_h,minimum_load_to_capacity_pct,maximum_load_to_capacity_pct
1,7,0.0,0.0,132.0,220.8,352.8,0.0,0.0
2,8,0.0,0.0,158.4,194.4,352.8,0.0,0.0
3,6,69.38,199.17,105.6,146.4,252.0,27.53,79.04
4,7,80.95,232.37,132.0,220.8,352.8,22.94,65.87
5,7,80.95,232.37,132.0,220.8,352.8,22.94,65.87
6,9,104.08,298.76,105.6,196.8,302.4,34.42,98.8
7,28,431.72,1239.29,528.0,883.2,1411.2,30.59,87.82
8,29,447.14,1283.56,501.6,960.0,1461.6,30.59,87.82


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 30:
-- Packing-Machine Eligibility Concentration
--
-- Business Question:
-- Which packing machines are eligible for the largest
-- number of demand-bearing products, and how much
-- planned demand is associated with those eligible routes?
--
-- Important:
-- Route-associated demand is NOT assigned workload.
-- A product mapped to multiple packing machines will
-- appear against each eligible machine.
--
-- Therefore, demand must NOT be summed across machines
-- and interpreted as total demand.
--
-- SQL Skills:
-- Multiple CTEs, JOIN, COUNT DISTINCT,
-- SUM, window functions, ROUND

WITH product_demand AS (

    SELECT
        product,

        SUM(adjusted_total_demand_m)
            AS total_planned_demand_m

    FROM workspace.gold_sop.fact_demand_plan

    GROUP BY
        product
),

machine_exposure AS (

    SELECT
        bpp.packing_machine,

        COUNT(DISTINCT bpp.product)
            AS eligible_demand_products,

        SUM(d.total_planned_demand_m)
            AS route_associated_demand_m

    FROM workspace.gold_sop.bridge_product_packing AS bpp

    INNER JOIN product_demand AS d
        ON bpp.product = d.product

    GROUP BY
        bpp.packing_machine
)

SELECT
    packing_machine,

    eligible_demand_products,

    route_associated_demand_m,

    ROUND(
        100.0 * route_associated_demand_m
        / SUM(route_associated_demand_m) OVER (),
        2
    ) AS share_of_route_associated_demand_pct

FROM machine_exposure

ORDER BY
    route_associated_demand_m DESC,
    packing_machine;

packing_machine,eligible_demand_products,route_associated_demand_m,share_of_route_associated_demand_pct
926,38,2724705,30.99
921,35,2657508,30.23
924,16,1186435,13.50
970,15,1035116,11.77
925,11,880815,10.02
927,1,282207,3.21
922,1,24480,0.28


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 31:
-- Packing-Machine Capacity Profile
--
-- Business Question:
-- How much regular, overtime, and total available
-- capacity does each packing machine have across
-- the complete planning horizon?
--
-- Important:
-- These are AVAILABLE capacity hours.
-- They are NOT actual utilized packing hours.
--
-- SQL Skills:
-- JOIN, COUNT DISTINCT, SUM,
-- arithmetic, ROUND, GROUP BY

SELECT
    pc.packing_machine,

    COUNT(DISTINCT pc.period)
        AS planning_periods,

    ROUND(
        SUM(pc.available_regular_time_h),
        2
    ) AS regular_capacity_h,

    ROUND(
        SUM(pc.available_overtime_h),
        2
    ) AS overtime_capacity_h,

    ROUND(
        SUM(
            pc.available_regular_time_h
            + pc.available_overtime_h
        ),
        2
    ) AS total_available_capacity_h,

    ROUND(
        100.0 * SUM(pc.available_overtime_h)
        / NULLIF(
            SUM(
                pc.available_regular_time_h
                + pc.available_overtime_h
            ),
            0
        ),
        2
    ) AS overtime_share_of_capacity_pct

FROM workspace.gold_sop.fact_packing_capacity AS pc

GROUP BY
    pc.packing_machine

ORDER BY
    total_available_capacity_h DESC,
    pc.packing_machine;

packing_machine,planning_periods,regular_capacity_h,overtime_capacity_h,total_available_capacity_h,overtime_share_of_capacity_pct
921,8,326.4,364.8,691.2,52.78
922,8,163.2,528.0,691.2,76.39
924,8,326.4,364.8,691.2,52.78
925,8,326.4,364.8,691.2,52.78
926,8,326.4,364.8,691.2,52.78
927,8,163.2,528.0,691.2,76.39
970,8,163.2,528.0,691.2,76.39


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 32:
-- Packing Route Processing-Time Profile by Machine
--
-- Business Question:
-- For each packing machine, how broad is its
-- demand-product eligibility and what packing-time
-- range is represented across those eligible products?
--
-- Important:
-- Eligibility does NOT represent actual allocation.
-- Packing times are source-supported route attributes.
--
-- SQL Skills:
-- CTE, JOIN, COUNT DISTINCT,
-- MIN, MAX, AVG, ROUND, GROUP BY

WITH demand_products AS (

    SELECT DISTINCT
        product

    FROM workspace.gold_sop.fact_demand_plan
)

SELECT
    bpp.packing_machine,

    COUNT(DISTINCT bpp.product)
        AS eligible_demand_products,

    ROUND(
        MIN(bpp.packing_time_sec_m),
        2
    ) AS minimum_packing_time_sec_m,

    ROUND(
        AVG(bpp.packing_time_sec_m),
        2
    ) AS average_packing_time_sec_m,

    ROUND(
        MAX(bpp.packing_time_sec_m),
        2
    ) AS maximum_packing_time_sec_m,

    ROUND(
        MAX(bpp.packing_time_sec_m)
        - MIN(bpp.packing_time_sec_m),
        2
    ) AS packing_time_range_sec_m

FROM workspace.gold_sop.bridge_product_packing AS bpp

INNER JOIN demand_products AS d
    ON bpp.product = d.product

GROUP BY
    bpp.packing_machine

ORDER BY
    bpp.packing_machine;

packing_machine,eligible_demand_products,minimum_packing_time_sec_m,average_packing_time_sec_m,maximum_packing_time_sec_m,packing_time_range_sec_m
921,35,1.2,1.27,1.5,0.3
922,1,5.7,5.7,5.7,0.0
924,16,2.7,2.7,2.7,0.0
925,11,0.9,1.51,1.6,0.7
926,38,1.7,1.81,2.1,0.4
927,1,21.7,21.7,21.7,0.0
970,15,2.1,2.1,2.1,0.0


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 33:
-- Management Planning Exceptions Summary
--
-- Business Question:
-- What are the most decision-relevant planning exceptions
-- identified by the source-supported SQL analysis?
--
-- Purpose:
-- Consolidate major production and packing planning signals
-- into a compact management-facing exception summary.
--
-- Important:
-- These are analytical planning signals.
-- They are NOT actual historical utilization,
-- confirmed shortages, or executed production schedules.
--
-- SQL Skills:
-- Multiple CTEs, UNION ALL, CASE,
-- aggregation, filtering, ROUND

WITH
-- ============================================================
-- 1. MACHINE 316 SINGLE-MACHINE PRODUCTION LOAD
-- ============================================================

machine_316_products AS (

    SELECT
        bpp.product,
        MIN(bam.production_time_sec_m)
            AS production_time_sec_m

    FROM workspace.gold_sop.bridge_product_production AS bpp

    INNER JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1

    GROUP BY
        bpp.product

    HAVING COUNT(DISTINCT bam.machine) = 1
       AND MIN(bam.machine) = 316
),

machine_316_period_load AS (

    SELECT
        d.period,

        SUM(
            d.adjusted_total_demand_m
            * p.production_time_sec_m
            / 3600.0
        ) AS theoretical_processing_h

    FROM workspace.gold_sop.fact_demand_plan AS d

    INNER JOIN machine_316_products AS p
        ON d.product = p.product

    GROUP BY
        d.period
),

machine_316_exceptions AS (

    SELECT
        c.period,

        l.theoretical_processing_h,

        c.available_regular_time_h,

        c.available_overtime_h,

        c.available_regular_time_h
        + c.available_overtime_h
            AS total_available_capacity_h,

        l.theoretical_processing_h
        - (
            c.available_regular_time_h
            + c.available_overtime_h
        ) AS capacity_gap_h

    FROM machine_316_period_load AS l

    INNER JOIN workspace.gold_sop.fact_machine_capacity AS c
        ON l.period = c.period
       AND c.machine = 316

    WHERE l.theoretical_processing_h
          > c.available_regular_time_h
            + c.available_overtime_h
),

-- ============================================================
-- 2. DEMAND WITHOUT CODE-1 PRODUCTION MAPPING
-- ============================================================

demand_products AS (

    SELECT
        product,
        SUM(adjusted_total_demand_m)
            AS total_planned_demand_m

    FROM workspace.gold_sop.fact_demand_plan

    GROUP BY
        product
),

code_1_products AS (

    SELECT DISTINCT
        product

    FROM workspace.gold_sop.bridge_product_production

    WHERE main_alternative = 1
),

unmapped_production_demand AS (

    SELECT
        d.product,
        d.total_planned_demand_m

    FROM demand_products AS d

    LEFT JOIN code_1_products AS p
        ON d.product = p.product

    WHERE p.product IS NULL
),

-- ============================================================
-- 3. PACKING ROUTING STRUCTURE
-- ============================================================

packing_route_profile AS (

    SELECT
        d.product,

        COUNT(DISTINCT bpp.packing_machine)
            AS eligible_packing_machines,

        COUNT(DISTINCT bpp.packing_time_sec_m)
            AS distinct_packing_times

    FROM demand_products AS d

    LEFT JOIN workspace.gold_sop.bridge_product_packing AS bpp
        ON d.product = bpp.product

    GROUP BY
        d.product
),

-- ============================================================
-- 4. PACKING PERIOD SCENARIO
-- ============================================================

product_packing_profile AS (

    SELECT
        product,

        MIN(packing_time_sec_m)
            AS fastest_packing_time_sec_m,

        MAX(packing_time_sec_m)
            AS slowest_packing_time_sec_m

    FROM workspace.gold_sop.bridge_product_packing

    GROUP BY
        product
),

period_packing_load AS (

    SELECT
        d.period,

        SUM(
            d.adjusted_total_demand_m
            * p.fastest_packing_time_sec_m
            / 3600.0
        ) AS minimum_theoretical_packing_h,

        SUM(
            d.adjusted_total_demand_m
            * p.slowest_packing_time_sec_m
            / 3600.0
        ) AS maximum_theoretical_packing_h

    FROM workspace.gold_sop.fact_demand_plan AS d

    INNER JOIN product_packing_profile AS p
        ON d.product = p.product

    GROUP BY
        d.period
),

packing_capacity AS (

    SELECT
        period,

        SUM(
            available_regular_time_h
            + available_overtime_h
        ) AS total_available_capacity_h

    FROM workspace.gold_sop.fact_packing_capacity

    GROUP BY
        period
),

packing_scenario AS (

    SELECT
        l.period,

        l.minimum_theoretical_packing_h,

        l.maximum_theoretical_packing_h,

        c.total_available_capacity_h,

        100.0
        * l.maximum_theoretical_packing_h
        / NULLIF(c.total_available_capacity_h, 0)
            AS maximum_load_to_capacity_pct

    FROM period_packing_load AS l

    INNER JOIN packing_capacity AS c
        ON l.period = c.period
),

-- ============================================================
-- 5. MANAGEMENT EXCEPTION OUTPUT
-- ============================================================

management_exceptions AS (

    SELECT
        'Production Capacity'
            AS exception_category,

        'Machine 316 total-capacity gap'
            AS exception_type,

        CAST(period AS STRING)
            AS planning_period,

        'Machine 316'
            AS affected_resource,

        ROUND(capacity_gap_h, 2)
            AS primary_metric,

        'Hours above total available capacity'
            AS metric_definition,

        'Review production routing and capacity options'
            AS management_action

    FROM machine_316_exceptions


    UNION ALL


    SELECT
        'Production Mapping'
            AS exception_category,

        'Demand product without code-1 production mapping'
            AS exception_type,

        'All demand periods'
            AS planning_period,

        CONCAT('Product ', CAST(product AS STRING))
            AS affected_resource,

        ROUND(total_planned_demand_m, 2)
            AS primary_metric,

        'Planned demand (m)'
            AS metric_definition,

        'Review source-supported production mapping'
            AS management_action

    FROM unmapped_production_demand


    UNION ALL


    SELECT
        'Packing Routing'
            AS exception_category,

        'Demand products with multiple packing routes'
            AS exception_type,

        'All demand periods'
            AS planning_period,

        'Packing network'
            AS affected_resource,

        COUNT(*)
            AS primary_metric,

        'Demand-bearing products'
            AS metric_definition,

        'Evaluate packing-machine allocation scenario'
            AS management_action

    FROM packing_route_profile

    WHERE eligible_packing_machines > 1
      AND distinct_packing_times > 1


    UNION ALL


    SELECT
        'Packing Capacity'
            AS exception_category,

        'Highest slowest-route packing scenario'
            AS exception_type,

        CAST(period AS STRING)
            AS planning_period,

        'Packing network'
            AS affected_resource,

        ROUND(maximum_load_to_capacity_pct, 2)
            AS primary_metric,

        'Slowest-route theoretical load / total capacity (%)'
            AS metric_definition,

        'Evaluate route allocation before concluding feasibility'
            AS management_action

    FROM packing_scenario

    WHERE maximum_load_to_capacity_pct = (
        SELECT MAX(maximum_load_to_capacity_pct)
        FROM packing_scenario
    )
)

SELECT
    exception_category,
    exception_type,
    planning_period,
    affected_resource,
    primary_metric,
    metric_definition,
    management_action

FROM management_exceptions

ORDER BY
    CASE exception_category
        WHEN 'Production Capacity' THEN 1
        WHEN 'Production Mapping' THEN 2
        WHEN 'Packing Routing' THEN 3
        WHEN 'Packing Capacity' THEN 4
        ELSE 5
    END,
    planning_period,
    affected_resource;

exception_category,exception_type,planning_period,affected_resource,primary_metric,metric_definition,management_action
Production Capacity,Machine 316 total-capacity gap,3,Machine 316,6.44,Hours above total available capacity,Review production routing and capacity options
Production Capacity,Machine 316 total-capacity gap,6,Machine 316,24.06,Hours above total available capacity,Review production routing and capacity options
Production Capacity,Machine 316 total-capacity gap,7,Machine 316,69.93,Hours above total available capacity,Review production routing and capacity options
Production Capacity,Machine 316 total-capacity gap,8,Machine 316,72.43,Hours above total available capacity,Review production routing and capacity options
Production Mapping,Demand product without code-1 production mapping,All demand periods,Product 7202031,25413.0,Planned demand (m),Review source-supported production mapping
Packing Routing,Demand products with multiple packing routes,All demand periods,Packing network,41.0,Demand-bearing products,Evaluate packing-machine allocation scenario
Packing Capacity,Highest slowest-route packing scenario,6,Packing network,98.8,Slowest-route theoretical load / total capacity (%),Evaluate route allocation before concluding feasibility


In [0]:
%sql
-- Project #4 — Supply Chain Planning & Operations Intelligence
-- SQL Analytics — Step 34:
-- SQL Analytics Completion Validation
--
-- Purpose:
-- Reconcile the major analytical findings generated
-- throughout the SQL notebook before closing the SQL phase.
--
-- Important:
-- Results describe source-supported analytical conditions
-- and theoretical planning scenarios, not actual historical
-- utilization or executed production/packing schedules.

WITH

-- ============================================================
-- 1. GOLD MODEL INVENTORY
-- ============================================================

gold_table_counts AS (

    SELECT
        SUM(row_count) AS gold_records

    FROM (
        SELECT COUNT(*) AS row_count FROM workspace.gold_sop.dim_product
        UNION ALL
        SELECT COUNT(*) FROM workspace.gold_sop.dim_planning_period
        UNION ALL
        SELECT COUNT(*) FROM workspace.gold_sop.dim_machine
        UNION ALL
        SELECT COUNT(*) FROM workspace.gold_sop.dim_packing_machine
        UNION ALL
        SELECT COUNT(*) FROM workspace.gold_sop.fact_customer_order
        UNION ALL
        SELECT COUNT(*) FROM workspace.gold_sop.fact_demand_plan
        UNION ALL
        SELECT COUNT(*) FROM workspace.gold_sop.fact_machine_capacity
        UNION ALL
        SELECT COUNT(*) FROM workspace.gold_sop.fact_packing_capacity
        UNION ALL
        SELECT COUNT(*) FROM workspace.gold_sop.bridge_product_production
        UNION ALL
        SELECT COUNT(*) FROM workspace.gold_sop.bridge_alternative_machine
        UNION ALL
        SELECT COUNT(*) FROM workspace.gold_sop.bridge_product_packing
        UNION ALL
        SELECT COUNT(*) FROM workspace.gold_sop.bridge_product_raw_material
    )
),

-- ============================================================
-- 2. DEMAND PROFILE
-- ============================================================

demand_profile AS (

    SELECT
        COUNT(DISTINCT period)
            AS demand_bearing_periods,

        COUNT(DISTINCT product)
            AS demand_bearing_products

    FROM workspace.gold_sop.fact_demand_plan
),

-- ============================================================
-- 3. CODE-1 PRODUCTION MAPPING COVERAGE
-- ============================================================

demand_products AS (

    SELECT
        product,
        SUM(adjusted_total_demand_m)
            AS total_planned_demand_m

    FROM workspace.gold_sop.fact_demand_plan

    GROUP BY product
),

code_1_products AS (

    SELECT DISTINCT
        product

    FROM workspace.gold_sop.bridge_product_production

    WHERE main_alternative = 1
),

production_mapping AS (

    SELECT
        COUNT(*) AS total_demand_products,

        SUM(
            CASE
                WHEN c.product IS NOT NULL THEN 1
                ELSE 0
            END
        ) AS mapped_demand_products,

        SUM(
            CASE
                WHEN c.product IS NULL THEN 1
                ELSE 0
            END
        ) AS unmapped_demand_products

    FROM demand_products AS d

    LEFT JOIN code_1_products AS c
        ON d.product = c.product
),

-- ============================================================
-- 4. MACHINE 316 SINGLE-MACHINE LOAD
-- ============================================================

machine_316_products AS (

    SELECT
        bpp.product,

        MIN(bam.production_time_sec_m)
            AS production_time_sec_m

    FROM workspace.gold_sop.bridge_product_production AS bpp

    INNER JOIN workspace.gold_sop.bridge_alternative_machine AS bam
        ON bpp.alternative = bam.alternative

    WHERE bpp.main_alternative = 1

    GROUP BY
        bpp.product

    HAVING COUNT(DISTINCT bam.machine) = 1
       AND MIN(bam.machine) = 316
),

machine_316_load AS (

    SELECT
        d.period,

        SUM(
            d.adjusted_total_demand_m
            * p.production_time_sec_m
            / 3600.0
        ) AS theoretical_processing_h

    FROM workspace.gold_sop.fact_demand_plan AS d

    INNER JOIN machine_316_products AS p
        ON d.product = p.product

    GROUP BY
        d.period
),

machine_316_gap AS (

    SELECT
        l.period,

        l.theoretical_processing_h
        - (
            c.available_regular_time_h
            + c.available_overtime_h
        ) AS capacity_gap_h

    FROM machine_316_load AS l

    INNER JOIN workspace.gold_sop.fact_machine_capacity AS c
        ON l.period = c.period
       AND c.machine = 316

    WHERE l.theoretical_processing_h
          >
          c.available_regular_time_h
          + c.available_overtime_h
),

machine_316_summary AS (

    SELECT
        COUNT(*) AS capacity_gap_periods,

        ROUND(MAX(capacity_gap_h), 2)
            AS largest_capacity_gap_h

    FROM machine_316_gap
),

-- ============================================================
-- 5. PACKING ROUTING PROFILE
-- ============================================================

packing_route_profile AS (

    SELECT
        d.product,

        COUNT(DISTINCT bpp.packing_machine)
            AS eligible_packing_machines,

        COUNT(DISTINCT bpp.packing_time_sec_m)
            AS distinct_packing_times

    FROM demand_products AS d

    LEFT JOIN workspace.gold_sop.bridge_product_packing AS bpp
        ON d.product = bpp.product

    GROUP BY
        d.product
),

packing_summary AS (

    SELECT
        SUM(
            CASE
                WHEN eligible_packing_machines > 1
                 AND distinct_packing_times > 1
                THEN 1
                ELSE 0
            END
        ) AS multi_route_demand_products

    FROM packing_route_profile
)

-- ============================================================
-- 6. FINAL COMPLETION VALIDATION
-- ============================================================

SELECT
    12 AS gold_analytical_tables,

    g.gold_records,

    (SELECT COUNT(*)
     FROM workspace.gold_sop.dim_planning_period)
        AS planning_periods,

    d.demand_bearing_periods,

    d.demand_bearing_products,

    p.mapped_demand_products
        AS demand_products_with_code_1_mapping,

    p.unmapped_demand_products
        AS demand_products_without_code_1_mapping,

    m.capacity_gap_periods
        AS machine_316_capacity_gap_periods,

    m.largest_capacity_gap_h
        AS machine_316_largest_gap_h,

    ps.multi_route_demand_products
        AS demand_products_with_multiple_packing_routes

FROM gold_table_counts AS g

CROSS JOIN demand_profile AS d
CROSS JOIN production_mapping AS p
CROSS JOIN machine_316_summary AS m
CROSS JOIN packing_summary AS ps;

gold_analytical_tables,gold_records,planning_periods,demand_bearing_periods,demand_bearing_products,demand_products_with_code_1_mapping,demand_products_without_code_1_mapping,machine_316_capacity_gap_periods,machine_316_largest_gap_h,demand_products_with_multiple_packing_routes
12,2828,8,6,41,40,1,4,72.43,41
